# 🥔 Demo: detección de enfermedades en hojas de papa (ViT-S/16 + Grad-CAM)
Proyecto final MIA-203 (UNI FIIS). Este notebook **descarga los pesos del modelo final y hace inferencia de principio a fin**. No necesita el dataset ni GPU (funciona en CPU).

1. Clona el repositorio e instala dependencias.
2. Descarga los pesos del modelo final (ViT-S/16 preentrenado, elegido por macro-F1 de validación).
3. Clasifica las 7 imágenes de ejemplo (1 por clase, del conjunto de prueba) y muestra el mapa Grad-CAM.
4. Clasifica **una foto propia**.
5. (Opcional) abre la app web interactiva con enlace público.

Repositorio: https://github.com/gabo2607/papa-cnn-vs-vit

In [ ]:
!git clone -q https://github.com/gabo2607/papa-cnn-vs-vit.git
%cd papa-cnn-vs-vit
!pip -q install timm==1.0.30 gradio==5.50.0

## 1. Cargar el modelo final (descarga automática de los pesos, ≈83 MB)

In [ ]:
import sys, json, torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
sys.path.insert(0, 'src')
from demo_app import cargar_modelo, preparar, NOMBRES
from gradcam import superponer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg, meta, clases, model, cam = cargar_modelo('modelo_final', device)
print('Modelo:', meta['run'], '| criterio:', meta['criterio'])
print(f"Validación macro-F1 {meta['val']['macro_f1']:.3f} | Prueba macro-F1 {meta['test']['macro_f1']:.3f} | accuracy {meta['test']['accuracy']:.3f}")

def diagnosticar(img):
    x, arr = preparar(img, cfg['img_size'])
    mapa, logits = cam(x.to(device))
    prob = logits.float().softmax(1)[0].cpu().numpy()
    return prob, arr, superponer(arr, mapa[0])

## 2. Ejemplos del conjunto de prueba (imágenes nunca vistas en el entrenamiento)

In [ ]:
ejemplos = sorted(Path('demo/ejemplos').glob('*.jpg'))
fig, ax = plt.subplots(2, len(ejemplos), figsize=(3 * len(ejemplos), 6.4))
for j, f in enumerate(ejemplos):
    prob, arr, ov = diagnosticar(Image.open(f))
    k = int(prob.argmax())
    ax[0, j].imshow(arr); ax[0, j].set_title(f'real: {f.stem}', fontsize=9)
    ax[1, j].imshow(ov); ax[1, j].set_title(f'pred: {clases[k]} ({prob[k]:.0%})', fontsize=9,
                                            color='green' if clases[k] == f.stem else 'red')
for a in ax.flat: a.axis('off')
plt.suptitle('Fila superior: foto | fila inferior: Grad-CAM (rojo = zonas que más pesaron en la decisión)')
plt.tight_layout(); plt.show()

## 3. Probar con una foto propia
Suba una foto de **hoja de papa** tomada de cerca (como las del dataset). La imagen se recorta al centro en formato cuadrado.

In [ ]:
from google.colab import files
subidas = files.upload()
for nombre in subidas:
    img = Image.open(nombre)
    prob, arr, ov = diagnosticar(img)
    orden = prob.argsort()[::-1][:3]
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(arr); ax[1].imshow(ov)
    for a in ax: a.axis('off')
    ax[0].set_title(nombre, fontsize=9); ax[1].set_title('Grad-CAM', fontsize=9)
    plt.show()
    for k in orden:
        print(f'{NOMBRES[clases[k]]:<40} {prob[k]:6.1%}')
    if prob[orden[0]] < 0.6:
        print('⚠️ Confianza baja: la foto puede diferir de las de entrenamiento (hoja de papa en campo, de cerca).')

## 4. (Opcional) App web interactiva
Crea un enlace público temporal (`*.gradio.live`) que se puede abrir desde cualquier navegador o celular. Detenga la celda para cerrarla.

In [ ]:
from demo_app import construir_app
construir_app('modelo_final', device).launch(share=True, debug=True)